# 🧪 Agent Provider Test

### 🔁 Step 1: Reset and Seed Agent Providers

In [1]:
from pathlib import Path
import os, sys

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_scores import seed_score_providers
from app.db.seeders.seed_tools import seed_tool_providers
from app.db.seeders.seed_prompt_providers import seed_prompt_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_context_providers import seed_context_providers
from app.db.seeders.seed_agent_engine_providers import seed_agent_engine_providers
from app.db.seeders.seed_agent_providers import seed_agent_providers
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_prompt_providers(session)
    seed_tool_providers(session)
    seed_score_providers(session)
    seed_context_providers(session)
    seed_agent_engine_providers(session)
    seed_agent_providers(session)

Working directory is now: C:\Repos\codecritic
Seeded AgentPrompt GUID: 817a9787-e8c4-45c6-a1f8-2cea6a82eeb5
Seeded SystemPrompt GUID: f7bb1f81-16cf-4c42-9b12-a320faac26ac
Seeded prompt provider configurations successfully.
Seeded tool configurations successfully.
Seeded score providers successfully.
Seeded context provider configurations successfully.
Seeded agent engine configurations successfully.
Seeded agent provider configurations successfully.


### 🔍 Step 2: Retrieve Agent Provider ID from DB

In [2]:
from sqlalchemy import select
from app.db.models import AgentProviderConfig

with Session(bind=engine) as session:
    record = session.execute(
        select(AgentProviderConfig).where(AgentProviderConfig.name == "Basic Agent Provider")
    ).scalar_one()
    agent_provider_id = record.id
    print("✅ Found Agent Provider ID:", agent_provider_id)

✅ Found Agent Provider ID: 1


### 🏗️ Step 3: Instantiate Agent Provider from Factory

In [3]:
from app.factories.agent_provider_factory import AgentProviderFactory

provider = AgentProviderFactory.create(agent_provider_id)
print("✅ AgentProvider instantiated:", provider.__class__.__name__)

✅ AgentProvider instantiated: BasicAgentProvider


### 🧪 Step 4: Run Agent Provider with Test Config

In [4]:
response = provider.run(agent_config={"goal": "test"}, experiment_id="test_exp_001", round=1)
print("✅ AgentProvider Response:", response)

✅ AgentProvider Response: [Agent] Executed with config: {'goal': 'test'}


### 📜 Step 5: Check Agent Provider Logs

In [5]:
from sqlalchemy import text

with engine.connect() as conn:
    rows = conn.execute(
        text("SELECT * FROM agent_provider_log WHERE experiment_id='test_exp_001'")
    ).fetchall()

assert rows, "❌ No agent provider logs found."
print("✅ Logged Agent Executions:")
for row in rows:
    print(row)

✅ Logged Agent Executions:
(1, 'test_exp_001', 1, '{"goal": "test"}', "[Agent] Executed with config: {'goal': 'test'}", 1, None, '2025-05-24T21:12:37.266836+00:00')
